# Evaluating QueryBot with pydantic_evals

Demonstrates writing a `pydantic_evals` `Dataset` against a live QueryBot: a task
function that runs one question through a fresh bot, `Evaluator`s that inspect the
response status and trace, and a `pass_rate()` helper that repeats the dataset several
times for confidence, since a live model is non-deterministic.

This is the live-model companion to `tests/evals/` (this repo's CI-safe reference
harness, driven by scripted `FunctionModel`s with no live API calls). Point this
notebook's pattern at your own spec catalog and questions to measure whether your agent
actually behaves the way you expect — `tests/evals/` only proves the trace/result-store/
evaluator wiring is consumable, not that any given model selects the right tool or
refuses appropriately.

Prerequisites
-------------
1. Create the DuckDB database (one-time setup): `python examples/data/setup_db.py`
2. Set your Anthropic API key: `export ANTHROPIC_API_KEY=sk-ant-...`
3. Install the agent-evals extra: `pip install aitaem[agent-evals]`

**Cost/runtime note:** `pass_rate(n=5)` below runs the 2-case dataset 5 times against a
live model — 10 bot invocations total. Each `bot.ask()` call is a multi-step tool loop
(`record_intent` → `resolve_intent` → `compute_metrics`, each a separate model turn), so
the actual number of model calls is meaningfully higher than 10. Budget accordingly
before running that cell.

Set the path to the local folder which contains your AITAEM examples. The following
sub-folders are needed inside `aitaem_folder_path`:
- `examples/data`
- `examples/metrics`
- `examples/slices`

In [ ]:
aitaem_folder_path = "/path/to/aitaem"  # e.g. "/Users/you/Workspace/aitaem"

In [2]:
import sys
import importlib

sys.path.insert(0, aitaem_folder_path + "/examples")
ex = importlib.import_module("04_evaluating_agents_example")

In [3]:
ex.check_api_key(exit_on_missing=False)

spec_cache, conn_mgr = ex.setup(base_path=aitaem_folder_path)
print(f"{len(spec_cache.metrics)} metrics: {', '.join(spec_cache.metrics)}")

query_task = ex.make_query_task(spec_cache, conn_mgr)

6 metrics: avg_revenue, campaign_count, ctr, max_revenue, roas, total_revenue


## Run the dataset once

A single run against a live model — useful for a quick look, but see the note on
`ToolSequenceIs` above before reading a single failed assertion as a bug.

In [4]:
report = await ex.dataset.evaluate(query_task, progress=False)
report.print(include_input=True, include_output=False)

                                    Evaluation Summary: query_task                                     
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Case ID               ┃ Inputs                                              ┃ Assertions ┃ Duration ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━┩
│ in_catalog_metric     │ QIn(question='What was total revenue in Q1 2024?')  │ ✔✗         │     8.7s │
├───────────────────────┼─────────────────────────────────────────────────────┼────────────┼──────────┤
│ out_of_catalog_metric │ QIn(question='What was sales velocity last month?') │ ✔✔         │     4.1s │
├───────────────────────┼─────────────────────────────────────────────────────┼────────────┼──────────┤
│ Averages              │                                                     │ 75.0% ✔    │     6.4s │
└───────────────────────┴─────────────────────────────────────────────────────┴────────────┴──────────┘

## Repeated-run confidence

10 bot invocations (5 reps × 2 cases), each a multi-turn tool loop — meaningfully more
than 10 model calls. Budget accordingly before running this cell.

In [ ]:
rates = await ex.pass_rate(ex.dataset, query_task, n=5)
ex.print_pass_rate(rates)